In [4]:
import pandas as pd
import json

# Load the CSV file
csv_file = "Bird Migration Data Final Version.csv"
df = pd.read_csv(csv_file)

# Drop rows with missing critical location/altitude data
df = df.dropna(subset=[
    "Start_Latitude", "Start_Longitude",
    "Min_Altitude_m", "Max_Altitude_m"
])

# Zone classification logic
def classify_zone(alt):
    if alt > 750:
        return "Red"
    elif alt > 400:
        return "Warning"
    else:
        return "Green"

# Add Zone_Class column
df["Zone_Class"] = df["Min_Altitude_m"].apply(classify_zone)

# Optional: Add other columns if needed (e.g., safe height, alt buffers)
df["Safe_Height_m"] = (df["Min_Altitude_m"] * 0.5).round()
df["Plane_Min_Alt"] = df["Min_Altitude_m"].apply(lambda x: "⚠️ Unsafe" if x - 500 < 0 else f"{int(x - 500)} m")
df["Plane_Max_Alt"] = df["Max_Altitude_m"].apply(lambda x: f"{int(x + 500)} m")

# Convert to GeoJSON
features = []
for _, row in df.iterrows():
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row["Start_Longitude"], row["Start_Latitude"]],
        },
        "properties": {
            "Bird_ID": row["Bird_ID"],
            "Species": row["Species"],
            "Zone_Class": row["Zone_Class"],
            "Safe_Height_m": row["Safe_Height_m"],
            "Plane_Min_Alt": row["Plane_Min_Alt"],
            "Plane_Max_Alt": row["Plane_Max_Alt"],
            "Migration_Start_Month": row.get("Migration_Start_Month"),
            "Migration_End_Month": row.get("Migration_End_Month"),
            "Tag_Type": row.get("Tag_Type"),
            "Flock_Size": row.get("Flock_Size"),
            "Origin": row.get("Origin"),
            "StartCountryName": row.get("StartCountryName"),
            "EndCountryName": row.get("EndCountryName")
            # Add more if needed
        },
    }
    features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "features": features
}

# Output file
with open("public/migration_data.geojson", "w", encoding="utf-8") as f:
    json.dump(geojson, f, indent=2)

print("✅ GeoJSON conversion complete: public/migration_data.geojson")

✅ GeoJSON conversion complete: public/migration_data.geojson
